In [1]:
# sentence-transformers turns text into embeddings (lists of numbers that capture meaning)
%pip install sentence-transformers

   ---------------------------------------- 0.0/760.7 kB ? eta -:--:--
   ---------------------------------------- 760.7/760.7 kB 8.0 MB/s  0:00:00
   ---------------------------------------- 0.0/857.0 kB ? eta -:--:--
   ---------------------------------------- 857.0/857.0 kB 12.5 MB/s  0:00:00
   ---------------------------------------- 0.0/3.9 MB ? eta -:--:--
   ------------------------ --------------- 2.4/3.9 MB 11.6 MB/s eta 0:00:01
   ---------------------------------------- 3.9/3.9 MB 11.1 MB/s  0:00:00
   ---------------------------------------- 0.0/13.1 MB ? eta -:--:--
   ------- -------------------------------- 2.4/13.1 MB 11.9 MB/s eta 0:00:01
   --------------- ------------------------ 5.0/13.1 MB 11.8 MB/s eta 0:00:01
   ---------------------- ----------------- 7.3/13.1 MB 11.8 MB/s eta 0:00:01
   ----------------------------- ---------- 9.7/13.1 MB 11.6 MB/s eta 0:00:01
   ------------------------------------- -- 12.3/13.1 MB 11.5 MB/s eta 0:00:01
   -------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [2]:
import json
import numpy as np                                     # for fast maths on lists of numbers
from sentence_transformers import SentenceTransformer  # the embedding model

# Load the 27 modules we saved in the extraction notebook
with open("modules.json", encoding="utf-8") as f:
    modules = json.load(f)

print(len(modules), "modules loaded")   # should print 27

c:\Users\ganes\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


27 modules loaded


In [3]:
# A free model that runs on your laptop and understands German AND English,
# so an English question can still find a German module.
# The first time, it downloads (~500 MB), so this can take a few minutes.
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

c:\Users\ganes\miniconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ganes\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\ganes\miniconda3\Lib\site-packages\huggingface_hub\file_download.py:149: U

In [4]:
# Take the cleaned text of each module
texts = [m["text"] for m in modules]

# Turn each text into a vector (a list of 384 numbers)
# normalize_embeddings=True scales them so we can compare them easily
module_vectors = model.encode(texts, normalize_embeddings=True)

print(module_vectors.shape)   # should print (27, 384): 27 modules, 384 numbers each

(27, 384)


In [5]:
def search(question, top_k=3):
    """Return the top_k modules most similar to the question."""
    # Turn the question into a vector the same way
    q_vector = model.encode([question], normalize_embeddings=True)[0]

    # Compare the question with all 27 modules at once.
    # @ is a dot product; for normalized vectors this is the cosine similarity.
    # Result: one score per module, higher = more similar
    scores = module_vectors @ q_vector

    # Sort the scores from high to low and keep the best top_k positions
    best = np.argsort(scores)[::-1][:top_k]

    # Return the module name, page and score for each of the best ones
    return [(modules[i]["name"], modules[i]["page"], round(float(scores[i]), 3)) for i in best]

# Try one question
search("Wer ist verantwortlich für Informationssysteme II?")

[('Informationssysteme I', 11, 0.572),
 ('Informationssysteme II', 23, 0.572),
 ('Wirtschaftsinformatik II', 22, 0.407)]

In [6]:
# Each test: (question, the module that should be found)
tests = [
    ("Who is responsible for Informationssysteme II?", "Informationssysteme II"),
    ("Wie viele CP hat das Projekt?", "Projekt"),
    ("Where do I learn SQL?", "Informationssysteme I"),
    ("Welches Modul behandelt Statistik?", "Wahrscheinlichkeitsrechnung und Statistik"),
    ("What do I need for the Bachelor thesis?", "Bachelorarbeit mit Bachelorkolloquium"),
]

hits = 0
for question, correct in tests:
    found = [name for name, page, score in search(question)]   # names of the top 3
    ok = correct in found                                       # is the right one in the top 3?
    hits += ok                                                  # True counts as 1
    print("OK  " if ok else "MISS", question, "->", found)

# Recall@3 = how often the right module was in the top 3
print(f"\nRecall@3: {hits} of {len(tests)}")

OK   Who is responsible for Informationssysteme II? -> ['Informationssysteme I', 'Informationssysteme II', 'Wirtschaftsinformatik II']
MISS Wie viele CP hat das Projekt? -> ['Wahrscheinlichkeitsrechnung und Statistik', 'Bachelorarbeit mit Bachelorkolloquium', 'Programmiermethodik II']
OK   Where do I learn SQL? -> ['Projekt', 'Seminar', 'Informationssysteme I']
OK   Welches Modul behandelt Statistik? -> ['Wahrscheinlichkeitsrechnung und Statistik', 'Quantitative Methoden', 'Informationssysteme I']
OK   What do I need for the Bachelor thesis? -> ['Seminar', 'Programmiermethodik I', 'Bachelorarbeit mit Bachelorkolloquium']

Recall@3: 4 of 5
